# M5 · Your first real fine-tune

In M4 you trained a model from random numbers. It took 2,000 steps to learn to spell. Real models start somewhere far better: someone has already trained them on trillions of tokens. **Fine-tuning** is taking a trained model and training it a little more, on your own examples, so it does one thing your way.

Your examples here teach a house style: rewriting jargon in plain English.

> *Pursuant to our returns policy, items must be returned in their original packaging within 30 days in order to be eligible for a refund.*
>
> *To get a refund, return items in their original box within 30 days.*

By the end of this lesson you'll have:

- seen what a real instruction-tuned model does with the task before any fine-tuning
- turned example pairs into training batches, with only the answers graded
- measured where the memory goes when every weight is trained
- fine-tuned a 494-million-parameter model on your Mac, and seen the difference
- scored the rewrites, and checked what the model forgot

**How to work through it** (same as M1 to M4)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

The first code cell downloads Qwen2.5-0.5B-Instruct (about 1 GB, once). Section 5 trains it for real: about a minute, and up to 24 GB of memory, so close other big apps first.

## 0. What fine-tuning changes

The model's weights after its makers trained it are called the **pretrained weights**. They already know grammar, facts and how to follow instructions. Train them a bit more on your examples and you get **fine-tuned weights**: the same model, nudged towards your task.

The training loop is the one you wrote in M4: batch, loss, `.backward()`, `optimizer.step()`. Three things are new:

1. **The examples are pairs**, a prompt and an answer, so the loss should grade only the answer (sections 2 and 3).
2. **The model is big**, and training every weight is expensive (section 4).
3. **The model already knows things**, and training on one narrow task can wear some of them away (section 8).

Training every single weight, as we do here, is a **full fine-tune**. Part C (M6 to M8) trains only a small number of extra weights instead, and this module shows you why.

## 1. The original model tries the task

Our model is **Qwen2.5-0.5B-Instruct**: 494 million parameters, already trained to follow instructions. Its chat format is ChatML, which you wrote by hand in M3.

We give it a system message with the instruction, and the jargon as the user's message.

In [ ]:
import json

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM = "Rewrite the user's text in plain English. Reply with the rewrite only."
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters, on {device}")


def read_jsonl(path):
    return [json.loads(line) for line in open(path)]


train_rows = read_jsonl("data/train.jsonl")
val_rows = read_jsonl("data/val.jsonl")
test_rows = read_jsonl("data/test.jsonl")
print(len(train_rows), len(val_rows), len(test_rows))
print(train_rows[0])

**What each line does**

- `AutoTokenizer.from_pretrained` and `AutoModelForCausalLM.from_pretrained`: the tokenizer and model from M2 and M3, downloaded once and then read from the cache.
- `dtype=torch.float32`: 4 bytes per number. We train in full 32-bit precision, so the memory sums in section 4 are simple.
- `.to(device)`: moves every weight to the GPU.
- `read_jsonl`: each line of a `.jsonl` file is one JSON object (M0). `json.loads` turns a line into a dict.
- 400 training pairs, 50 for validation and 50 for the final test. Each pair has a `jargon` side and a `plain` side.

In [ ]:
def prompt_ids(text, system=SYSTEM):
    messages = [{"role": "user", "content": text}]
    if system is not None:
        messages.insert(0, {"role": "system", "content": system})
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True)["input_ids"]


@torch.no_grad()
def reply(text, system=SYSTEM, max_new=80):
    model.eval()
    ids = torch.tensor([prompt_ids(text, system)], device=device)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
    model.train()
    return tokenizer.decode(out[0, ids.shape[1] :], skip_special_tokens=True).strip()


print(tokenizer.decode(prompt_ids("Let's touch base tomorrow.")))
for row in test_rows[:3]:
    print("jargon:  ", row["jargon"])
    print("original:", reply(row["jargon"]))
    print()

**What each line does**

- `prompt_ids`: the system message and the user's text, formatted with the chat template and turned into ids (M3 section 6). `add_generation_prompt=True` ends it with `<|im_start|>assistant`, so the model's reply comes next.
- `messages.insert(0, ...)`: puts the system message first. With `system=None` there's no instruction at all; section 8 uses that.
- `reply`: greedy generation (M2), then decodes only the new tokens: `out[0, ids.shape[1]:]` skips the prompt.
- The printed prompt shows exactly what the model receives, special tokens and all.
- The original model does rewrite, and drops a little jargon. But it keeps formal words ("objective", "implement", "qualify") and long sentences: it follows the instruction its own way, not your house style.

That's the usual starting point: an instruction-tuned model can roughly do your task already. Fine-tuning is for doing it *your* way, every time, without a long list of instructions.

Before we change any weights, let's also ask it some ordinary questions, with no system message. We'll ask again after fine-tuning (section 8).

In [ ]:
general = read_jsonl("data/general.jsonl")
before = {g["question"]: reply(g["question"], system=None, max_new=96) for g in general}
for g in general[:4]:
    print(g["question"], "->", before[g["question"]])
print(
    sum(g["expect"] in before[g["question"]].lower() for g in general), "of", len(general), "right"
)

**What each line does**

- `general`: 10 ordinary questions, each with the word or number a right answer must contain (`expect`).
- `before`: a dict comprehension, from each question to the original model's answer.
- `g["expect"] in answer.lower()`: `True` if the answer contains the expected text. `sum` of `True`s counts them, since `True` counts as 1.

## 2. Formatting the pairs, and padding a batch

Each training example is one sequence: the prompt (system message and jargon, formatted by the chat template) followed by the answer (the plain rewrite) and `<|im_end|>`, the token that ends the assistant's turn. Ending with `<|im_end|>` teaches the model to stop when the rewrite is done.

In [ ]:
END = tokenizer.convert_tokens_to_ids("<|im_end|>")


def answer_ids(text):
    return tokenizer.encode(text, add_special_tokens=False) + [END]


row = train_rows[0]
p, a = prompt_ids(row["jargon"]), answer_ids(row["plain"])
print(len(p), "prompt tokens,", len(a), "answer tokens")
print(repr(tokenizer.decode(p + a)))

**What each line does**

- `convert_tokens_to_ids("<|im_end|>")`: the id of the end-of-turn token (M3 section 5).
- `add_special_tokens=False`: just the text's own tokens, nothing added.
- `p + a`: `+` joins two lists into one longer list. That's the whole training sequence.
- The prompt is much longer than the answer: the system message alone is about 20 tokens.

Examples come in different lengths, but a batch has to be a rectangle. So we **pad**: add filler tokens to the end of the shorter rows (M3 section 5). For training, padding goes on the right; for generating, on the left (M3's question 6).

Three tensors make up the batch:

- `input_ids`: the tokens, padded with the pad token.
- `attention_mask`: 1 for real tokens, 0 for padding, so the model never looks at padding.
- `labels`: what each position is graded against. Padding gets the label **-100**, which cross-entropy reads as "don't grade this position".

**Predict:** for rows of length 3 and 2, what does the second row's attention mask look like?

In [ ]:
rows = [[11, 12, 13], [21, 22]]
pad_id = tokenizer.pad_token_id
longest = max(len(r) for r in rows)

input_ids, attention_mask = [], []
for r in rows:
    pad = longest - len(r)
    input_ids.append(r + [pad_id] * pad)
    attention_mask.append([1] * len(r) + [0] * pad)

print(pad_id, repr(tokenizer.decode([pad_id])))
print(torch.tensor(input_ids, dtype=torch.long))
print(torch.tensor(attention_mask, dtype=torch.long))

**What each line does**

- `max(len(r) for r in rows)`: the length of the longest row, 3.
- `[pad_id] * pad`: a list of `pad` copies of the pad id. `* 0` gives an empty list, so the longest row gets no padding.
- `[1] * len(r) + [0] * pad`: ones for the real tokens, then zeros for the padding: `[1, 1, 0]`.
- `torch.tensor(..., dtype=torch.long)`: lists of lists become a 2-D tensor of whole numbers. This only works because every row now has the same length.
- Qwen's pad token is `<|endoftext|>`. Labels get padded the same way, but with `-100`. That's exercise 2.

## 3. Loss masking: only the answer is graded

If we graded every position, the model would also be trained to write the system message and the user's jargon: text it will always be *given*, never asked to write. Worse, most of the tokens are prompt, so most of the training would go there.

**Loss masking** fixes this: in the labels, every prompt token becomes -100. The model still reads the whole sequence, but it is only graded on the answer.

In [ ]:
labels = [-100] * len(p) + a
print(len(p + a), "tokens,", sum(x != -100 for x in labels), "of them graded")
for tok, lab in list(zip(tokenizer.convert_ids_to_tokens(p + a), labels))[len(p) - 3 : len(p) + 3]:
    print(f"{tok!r:14} {lab}")

**What each line does**

- `[-100] * len(p) + a`: one -100 for each prompt token, then the answer's own ids. Same length as `p + a`. That's exercise 1.
- `sum(x != -100 for x in labels)`: counts the graded positions: only the answer's.
- `[len(p) - 3 : len(p) + 3]`: the three tokens either side of where the prompt ends. (`Ċ` is how this tokenizer shows a newline, like `Ġ` for a space in M3.) The loop shows that moment: `assistant` and the newline after it are still -100, and the answer's first token is the first one graded.

One more detail from M4: the model at position *t* guesses the token at *t* + 1. In M4 we built `y` already shifted. Here `labels` lines up with `input_ids`, so the loss has to do the shift: drop the last position's logits (there's nothing after it to guess) and the first label (nothing guesses it).

Hugging Face models do exactly this when you pass `labels=`. Let's check our version against theirs:

In [ ]:
ids = torch.tensor([p + a], device=device)
lab = torch.tensor([labels], device=device)
with torch.no_grad():
    logits = model(input_ids=ids).logits
    theirs = model(input_ids=ids, labels=lab).loss

shifted_logits = logits[:, :-1, :]
shifted_labels = lab[:, 1:]
vocab = shifted_logits.shape[-1]
ours = F.cross_entropy(
    shifted_logits.reshape(-1, vocab), shifted_labels.reshape(-1), ignore_index=-100
)
print(logits.shape, shifted_logits.shape, shifted_labels.shape)
print(ours.item(), theirs.item())

**What each line does**

- `model(input_ids=ids, labels=lab).loss`: the model computes its own loss when you give it labels.
- `logits[:, :-1, :]`: every position except the last. `lab[:, 1:]`: every label except the first. Now position *t*'s guess sits next to the token at *t* + 1.
- `ignore_index=-100`: skip every position whose label is -100. (-100 is actually the default, so leaving it out gives the same answer; writing it makes the intent clear.)
- The two losses match. That's exercise 3.

## 4. Where the memory goes

A full fine-tune keeps four things in memory for every weight:

| What | Bytes per weight (float32) |
|---|---|
| the weight itself | 4 |
| its gradient | 4 |
| AdamW's first running average (of the gradients) | 4 |
| AdamW's second running average (of the squared gradients) | 4 |

Those two running averages are the **optimizer state**. That's 16 bytes per weight, before **activations**: the in-between results of the forward pass, kept so `.backward()` can use them. Activations grow with batch size and sequence length.

**Predict:** 494 million weights x 16 bytes. How many GB?

In [ ]:
n = sum(p.numel() for p in model.parameters())
print(f"weights:         {n * 4 / 1e9:.2f} GB")
print(f"+ gradients:     {n * 4 / 1e9:.2f} GB")
print(f"+ AdamW's state: {n * 8 / 1e9:.2f} GB")
print(f"= {n * 16 / 1e9:.2f} GB before activations")

**What each line does**

- `p.numel()`: how many numbers are in one weight tensor. Summed over every tensor: 494 million.
- `n * 4 / 1e9`: bytes to GB. `1e9` is a billion.
- About 7.9 GB. Now let's take one real step and see what AdamW actually stores.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.0)
print("entries before a step:", len(optimizer.state))

model(input_ids=ids, labels=lab).loss.backward()
optimizer.step()
optimizer.zero_grad()

first = next(iter(optimizer.state.values()))
for name, value in first.items():
    print(name, tuple(value.shape), value.dtype, value.numel() * value.element_size(), "bytes")

state = 0
for entry in optimizer.state.values():
    for value in entry.values():
        if torch.is_tensor(value):
            state += value.numel() * value.element_size()
print(f"AdamW's state: {state / 1e9:.2f} GB")
if device == "mps":
    print(f"in use on the GPU now: {torch.mps.driver_allocated_memory() / 1e9:.1f} GB")

**What each line does**

- `len(optimizer.state)`: 0 before the first step. AdamW only makes its notes once it has seen a gradient.
- `optimizer.state`: a dict with one entry per weight tensor. `next(iter(...values()))` takes the first entry, just to look.
- Each entry holds `exp_avg` and `exp_avg_sq` (the two running averages, each the same shape as the weight) and `step` (a one-number counter).
- `value.numel() * value.element_size()`: how many numbers times bytes per number. `torch.is_tensor` skips anything that isn't a tensor. That's exercise 4.
- About 3.95 GB for AdamW alone, twice the weights. `driver_allocated_memory` is everything the GPU holds now. During training it peaks above 20 GB, because of the activations.

A 0.5B model fits. But the sum scales with the model: a 7B model needs about 112 GB before activations, three times this Mac's memory. That's the lesson of full fine-tuning on a laptop: it works for small models only. Part C gets around it by training far fewer weights.

## 5. Fine-tuning for real

Now the M4 loop, on the real model: one pass over the 400 pairs (one **epoch**), batches of 8, learning rate 1e-5. That's a hundred times smaller than M4's: the weights are already good, and big steps would wreck them.

We measure the loss on the validation pairs before and after. It's computed with the same masking, so it measures exactly how well the model predicts the plain rewrites it has never seen.

In [ ]:
def make_batch(rows):
    examples = [(prompt_ids(r["jargon"]), answer_ids(r["plain"])) for r in rows]
    longest = max(len(p) + len(a) for p, a in examples)
    ids, mask, labs = [], [], []
    for p, a in examples:
        pad = longest - len(p) - len(a)
        ids.append(p + a + [pad_id] * pad)
        mask.append([1] * (len(p) + len(a)) + [0] * pad)
        labs.append([-100] * len(p) + a + [-100] * pad)
    return [torch.tensor(x, dtype=torch.long, device=device) for x in (ids, mask, labs)]


@torch.no_grad()
def val_loss():
    model.eval()
    losses = []
    for i in range(0, len(val_rows), 10):
        ids, mask, labs = make_batch(val_rows[i : i + 10])
        losses.append(model(input_ids=ids, attention_mask=mask, labels=labs).loss.item())
    model.train()
    return sum(losses) / len(losses)


print("val loss before:", round(val_loss(), 3))
model.train()
order = torch.randperm(len(train_rows), generator=torch.Generator().manual_seed(0)).tolist()
for step, i in enumerate(range(0, len(train_rows), 8)):
    ids, mask, labs = make_batch([train_rows[j] for j in order[i : i + 8]])
    loss = model(input_ids=ids, attention_mask=mask, labels=labs).loss
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 10 == 0:
        print(f"  step {step:3d}  loss {loss.item():.3f}")
print("val loss after: ", round(val_loss(), 3))

**What each line does**

- `make_batch`: sections 2 and 3 together: format each pair, pad every row to the longest, mask the prompt and the padding. Exercises 1 and 2 split this into two functions.
- The last line turns the three lists into tensors on the GPU, and returns them as a list of three; `ids, mask, labs = make_batch(...)` unpacks them.
- `val_loss`: the average masked loss over the 50 validation pairs, 10 at a time, in evaluation mode with no gradients (M4 section 8).
- `torch.randperm(400, generator=...)`: the numbers 0 to 399 in a random order: a shuffle, so each batch is a random mix.
- The loop is M4's: loss, `zero_grad`, `backward`, clip, `step`. We pass `labels=` and let the model do the shift (section 3).
- The validation loss falls from about 1.8 to about 0.9 in 50 steps: the model now finds the plain rewrites far more likely. It starts at 1.8, not the 1.55 `train.py` reports for the original model, because of the single AdamW step we took in section 4: that one step moved every weight by about the learning rate, in a direction chosen from just one example, and made the model slightly worse.

In [ ]:
for row in test_rows[:3]:
    print("jargon:    ", row["jargon"])
    print("fine-tuned:", reply(row["jargon"]))
    print("reference: ", row["plain"])
    print()

**What each line does**

- The same three test sentences as section 1, which the model never trained on.
- Compare with section 1: shorter, plainer, and in the same voice as the training rewrites. That's one epoch, about 20 seconds.

## 6. The same run with `Trainer`

Writing the loop yourself is the best way to understand it. In practice, people mostly use Hugging Face's **`Trainer`**, which runs the same loop for you: batching, shuffling, the learning-rate schedule, clipping, evaluation and logging. You describe the run with **`TrainingArguments`** and hand over a **data collator**: the function that turns a list of examples into one padded batch. That's exactly exercise 2.

One difference from your loop: `Trainer` doesn't use a loss function you give it. It passes `labels=` to the model and uses the model's own shifted, masked loss (section 3). So in `train.py --trainer`, your `collate` builds every batch, but your `masked_lm_loss` isn't used.

To keep memory free, this demo uses a tiny random model with the same parts as Qwen (`tiny_model.py`), on the CPU. `train.py --trainer` does the real run.

In [ ]:
import tempfile

from tiny_model import make_tiny_model
from transformers import Trainer, TrainingArguments

tiny = make_tiny_model()
fake = [{"input_ids": [5, 6, 7, 8, 9], "labels": [-100, -100, -100, 8, 9]}] * 16


def collator(features):
    longest = max(len(f["input_ids"]) for f in features)

    def pad(x, v):
        return x + [v] * (longest - len(x))

    return {
        "input_ids": torch.tensor([pad(f["input_ids"], 0) for f in features]),
        "attention_mask": torch.tensor([pad([1] * len(f["input_ids"]), 0) for f in features]),
        "labels": torch.tensor([pad(f["labels"], -100) for f in features]),
    }


with tempfile.TemporaryDirectory() as tmp:
    args = TrainingArguments(
        output_dir=tmp,
        per_device_train_batch_size=4,
        num_train_epochs=2,
        learning_rate=3e-3,
        logging_steps=2,
        save_strategy="no",
        report_to=[],
        use_cpu=True,
        disable_tqdm=True,
        remove_unused_columns=False,
    )
    trainer = Trainer(model=tiny, args=args, train_dataset=fake, data_collator=collator)
    trainer.train()
print([round(h["loss"], 3) for h in trainer.state.log_history if "loss" in h])

**What each line does**

- `fake`: 16 copies of one tiny example, as dicts with `input_ids` and `labels`. `Trainer` passes lists of these to the collator.
- `collator`: pads a list of examples into a batch, with the -100 padding for labels. `pad` is a small function defined inside it, so it can use `longest`.
- `TrainingArguments(...)`: the settings: batch size, epochs, learning rate, how often to log. `save_strategy="no"` skips saving checkpoints; `report_to=[]` skips online logging services; `remove_unused_columns=False` keeps our dict keys as they are.
- `Trainer(...).train()`: runs the whole loop, printing a line each time it logs (the loss, the gradient norm and the learning rate, falling along its schedule). `log_history` keeps those lines; the loss falls as the tiny model memorises its one example.
- `tempfile.TemporaryDirectory()`: a folder that's deleted at the end of the `with` block, for anything `Trainer` wants to write.

## 7. Scoring the rewrites

"Looks better" isn't a measurement. The checkpoint uses the validation loss from section 5, and two scores on the generated rewrites.

**chrF** (character n-gram F-score) compares a rewrite with the reference rewrite, character by character. Remove the spaces, then cut both texts into pieces: every 1 character, every 2 characters in a row, and so on up to 6. For each size:

- **precision**: the share of the rewrite's pieces that also appear in the reference
- **recall**: the share of the reference's pieces that the rewrite has

A piece only matches as many times as it appears in the reference (that's "clipping"). Average precision and recall over the sizes, then combine them into one **F-score** that weights recall twice as much (beta = 2): 100 is identical, 0 is nothing in common.

In [ ]:
def pieces(text, n):
    counts = {}
    for i in range(len(text) - n + 1):
        g = text[i : i + n]
        counts[g] = counts.get(g, 0) + 1
    return counts


hyp, ref = "ab", "abc"
for n in (1, 2, 3):
    h, r = pieces(hyp, n), pieces(ref, n)
    if len(h) == 0 or len(r) == 0:
        print(n, "skipped: 'ab' has no pieces of 3 characters")
        continue
    matches = sum(min(c, r.get(g, 0)) for g, c in h.items())
    print(n, h, r, "precision", matches / sum(h.values()), "recall", matches / sum(r.values()))

prec, rec, beta = 1.0, (2 / 3 + 1 / 2) / 2, 2.0
print("F =", 100 * (1 + beta**2) * prec * rec / (beta**2 * prec + rec))
print(pieces("aaa", 1), pieces("a", 1), "clipped matches:", min(3, 1))
print(repr("We will talk".replace(" ", "")))

**What each line does**

- `pieces`: counts every run of `n` characters, with the same `counts.get(g, 0) + 1` as M3's `pair_counts`. `range(len(text) - n + 1)` stops where the last full piece starts.
- `min(c, r.get(g, 0))`: a piece counts as matched at most as often as the reference has it. `r.get(g, 0)` is 0 if the reference doesn't have it at all.
- For `ab` vs `abc`: size 1 has precision 1 and recall 2/3; size 2 has precision 1 and recall 1/2. Averaged and combined, F is about 63.6.
- Size 3: `ab` is too short to have any 3-character pieces. `continue` skips straight to the next time round the loop, so that size isn't counted at all.
- `"aaa"` against `"a"`: three `a`s, but only one can match. Without clipping, repeating a word would look like a better rewrite.
- `.replace(" ", "")`: removes the spaces first. That's exercise 5.

chrF is fuzzy: a rewrite can be perfectly plain and still use different words from the reference, and lose points. So the second score is simpler: how many **buzzwords** are left. `data/buzzwords.txt` lists 86 of them. To count them fairly, compare whole words, so "leveraged" doesn't count as "leverage": lowercase, turn every non-letter into a space, split into words, and slide a window as long as the buzzword along the text.

In [ ]:
def words(text):
    return "".join(c if c.isalpha() else " " for c in text.lower()).split()


text = "Let's circle back, and leverage best-in-class tools. Leveraged? No."
w = words(text)
print(w)
for buzzword in ["circle back", "leverage", "best-in-class"]:
    phrase = words(buzzword)
    n = len(phrase)
    hits = [i for i in range(len(w) - n + 1) if w[i : i + n] == phrase]
    print(buzzword, phrase, "found at", hits)

**What each line does**

- `c if c.isalpha() else " "`: keeps letters and turns everything else (spaces, commas, hyphens, apostrophes) into a space. `"".join(...)` glues the characters back into one string, and `.split()` cuts it into words, throwing away the extra spaces.
- `best-in-class` becomes `['best', 'in', 'class']`, both in the text and in the buzzword, so they match.
- `w[i : i + n] == phrase`: two lists are equal only if every word matches, in order. `leveraged` isn't `leverage`, so it doesn't count.
- That's exercise 6.

## 8. Forgetting

Fine-tuning changed every weight of the model to make it better at one thing. Did that cost it anything else? Let's ask the ordinary questions from section 1 again.

In [ ]:
after = {g["question"]: reply(g["question"], system=None, max_new=96) for g in general}
for g in general[:4]:
    print(g["question"], "->", after[g["question"]])
right = sum(g["expect"] in after[g["question"]].lower() for g in general)
print(
    right,
    "of",
    len(general),
    "right (before:",
    sum(g["expect"] in before[g["question"]].lower() for g in general),
    ")",
)

**What each line does**

- The same 10 questions, the same greedy generation, with no system message.
- The answers have changed. They're shorter and plainer (the house style leaks into everything), and some are now wrong. In the reference run of this notebook the score went from 10 to 8, and the Mars answer claimed the planet "has no atmosphere and is very hot". Your numbers may differ a little.

That's **catastrophic forgetting**: training only on one narrow task pulls the weights away from everything else they did. Nothing in the rewriting data was about planets, yet the model got worse at them.

It varies from run to run, and gets worse the harder you train. With `train.py`'s defaults (2 epochs, learning rate 1e-5), the reference runs scored 10 of 10 with the loop and 8 of 10 with `Trainer`. With 3 epochs at learning rate 2e-5, the model wrote perfect plain English but got 3 of the 10 wrong, and its validation loss went *up* after the first epoch (0.89, 1.10, 1.20): the overfitting you saw in M4.

Ways to limit it: fewer epochs, a lower learning rate, mixing some general examples into the training data, or (Part C) leaving the original weights frozen and training a small add-on instead.

`train.py` reports this check for every run. It isn't part of the pass mark, but it's worth reading.

## 9. Your task

You've now seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are six exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `build_example` (section 3)
   2. `collate` (section 2)
   3. `masked_lm_loss` (section 3)
   4. `optimizer_state_bytes` (section 4)
   5. `chrf` (section 7)
   6. `count_buzzwords` (section 7)
2. Check your work as you go: `make check M=05`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`. Predict first, then check yourself in this notebook.
4. Fine-tune twice (close this notebook first, or restart its kernel, to free its memory):
   - `uv run python modules/m05_full_finetune/train.py --loop`: your loop, with your exercises.
   - `uv run python modules/m05_full_finetune/train.py --trainer`: `Trainer`, with your `collate`.

   Each takes about 3 minutes: 2 epochs of training, then scoring the original and the fine-tuned model on the 50 test pairs. To pass, the held-out loss must fall by at least 0.4, the buzzwords per rewrite must not go up, and chrF must stay within 3 points of the original model's.
5. Run `make check M=05` again. When everything passes, M5 turns green on the dashboard.

Stuck? `make hint M=05 EX=5` gives a nudge for exercise 5. Every term is in the glossary in `README.md`.

**Stretch:** run `train.py --loop --epochs 3 --lr 2e-5` and compare the validation loss after each epoch and the ordinary-questions score with your first run. Then try `--epochs 1`.